# 10 — Frozen 128-record development replication

**CPU runtime → Runtime → Run all → upload your completed notebook 08 AND notebook 09b ZIPs together.**

This extends the earlier pilot to 128 already selected malicious development records, excluding the 32 pilot originals. It compares repair during MOEVA against repair after search using the same frozen models and constraints. It does not train an IDS or replay packets.

Leave `RUN_MODE` at **full** for the research experiment. The other modes are implementation checks only. Keep both source ZIPs and download the new result ZIP when the run finishes. Every session is new.

In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

RUN_MODE = "full" #@param ["full", "preflight", "smoke"]
RUN_MODE = os.environ.get("PHD_REPLICATION_MODE", RUN_MODE)
PILOT_ZIP = "" #@param {type:"string"}
SEMANTICS_ZIP = "" #@param {type:"string"}
PILOT_ZIP = os.environ.get("PHD_PILOT_ZIP", PILOT_ZIP)
SEMANTICS_ZIP = os.environ.get("PHD_SEMANTICS_ZIP", SEMANTICS_ZIP)
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
if os.environ.get("PHD_NO_DOWNLOAD") == "1":
    DOWNLOAD_RESULTS = False
if RUN_MODE not in ("full", "preflight", "smoke"):
    raise ValueError("Unknown run mode")

def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 ** 2), b""):
            h.update(block)
    return h.hexdigest()

REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
HOST_PYTHON, HOST_VERSION = sys.executable, sys.version
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/" + unique_name("MyPhDThesis_Followup128"))).resolve()
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError("Choose a new workspace or an existing project git checkout")
for name in ("baseline/run_frozen_followup.py", "scripts/summarize_frozen_followup.py", "notebooks/10_Frozen_128_Development_Replication.ipynb"):
    if not (ROOT / name).is_file():
        raise RuntimeError("Missing " + name + "; start with a fresh clone")
SESSION = ROOT / "runs" / unique_name("frozen_followup128")
SESSION.mkdir(parents=True, exist_ok=False)
source_paths = [p for folder in ("baseline", "scripts", "tests") for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/10_Frozen_128_Development_Replication.ipynb", ROOT / "docs/FROZEN_128_REPLICATION_GUIDE.md"]
context = dict(project_repository=REPO_URL,
    project_commit=subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    project_status=subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
    session_utc=dt.datetime.now(dt.timezone.utc).isoformat(), notebook_python=HOST_VERSION,
    notebook_executable=HOST_PYTHON, experiment="frozen_128_development_replication", mode=RUN_MODE,
    source_sha256={str(p.relative_to(ROOT)): file_hash(p) for p in source_paths})
with (SESSION / "session_context.json").open("x") as f:
    json.dump(context, f, indent=2); f.write("\n")
for p in source_paths:
    target = SESSION / "source_snapshot" / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("x") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path.name}. Run the last cell to export partial evidence.")

print("New session:", SESSION)
print("Run mode:", RUN_MODE, "| Project commit:", context["project_commit"])

## 1. Upload both reviewed result ZIPs

Select the completed **connection_search_20261013...zip** and **measurement_semantics_20261021T012235Z...zip** in the same upload dialog. ZIP names may change, but their content hashes must match the reviewed exports. The first preserves the scaler and split; the second contains the frozen 128 inputs.

In [ ]:
EXPECTED_PILOT = "bb1e0d9afe2eb4db9ed6b1a699b72a51d77abc86b27e00745a006daabf98ddd1"
EXPECTED_SEMANTICS = "e44eb9bf358bc7736e8e072a9b825b4dbc520ba36a7c2c05b50e397703ec076c"
if PILOT_ZIP.strip() and SEMANTICS_ZIP.strip():
    candidates = [Path(PILOT_ZIP).expanduser().resolve(), Path(SEMANTICS_ZIP).expanduser().resolve()]
elif PILOT_ZIP.strip() or SEMANTICS_ZIP.strip():
    raise ValueError("Provide both paths, or leave both empty for upload")
else:
    from google.colab import files
    upload_dir = SESSION / "upload"
    upload_dir.mkdir(exist_ok=False)
    previous_cwd = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    if len(uploaded) != 2:
        raise ValueError("Select exactly two files: notebook 08 and notebook 09b result ZIPs")
    candidates = [upload_dir / name for name in uploaded]
    del uploaded
if not all(p.is_file() and p.suffix.lower() == ".zip" for p in candidates):
    raise ValueError("Both inputs must be existing ZIP files")
by_hash = {file_hash(p): p for p in candidates}
if set(by_hash) != {EXPECTED_PILOT, EXPECTED_SEMANTICS}:
    raise ValueError("These are not the two reviewed exports. Keep the original completed 08 and 09b ZIPs.")
PILOT_INPUT, SEMANTICS_INPUT = by_hash[EXPECTED_PILOT], by_hash[EXPECTED_SEMANTICS]
print("Notebook 08:", PILOT_INPUT.name)
print("Notebook 09b:", SEMANTICS_INPUT.name)

## 2. Prepare the pinned model environment

The experiment uses isolated Python 3.8.20, NumPy 1.23.5 and PyTorch 1.12.1+cpu. The notebook's Python is unchanged. Setup uses the existing `uv` bootstrap and does not ask Colab's `venv` to run `ensurepip`. A GPU is not used by this fixed comparison.

A fresh setup downloads the pinned model dependencies, about 348 MB of feature data, and two small model checkpoints. The 128-record experiment is larger than the 32-record pilot; runtime depends on Colab's CPU allocation.

In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
shutil.copy2(ROOT / ".cache/setup_report.json", SESSION / (unique_name("setup_report") + ".json"))
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
shutil.copy2(ROOT / "assets/download_manifest.json", SESSION / (unique_name("download_manifest") + ".json"))

## 3. Run the fixed comparison

The runner verifies both input archives, all asset hashes, the original split and the frozen sample. It rechecks the 143,046 train/development controls and all 128 originals, then scores both frozen models. No native test rows are parsed or scored.

Full mode runs two models × three seeds × three generation variants (18 runs), producing five final methods per model/seed. The primary comparison is `caa_profile_relation_inloop` against `caa_profile_relation_end`. Budgets remain CAPGD 10 steps, MOEVA 100 generations with 50 offspring, and scaled L2 0.5. The packet/duration diagnostic rejected in notebook 09b is not added.

All originals remain in the output. Initially missed originals are reported separately and cannot count as new evasions. Before-MOEVA arrays, stage membership, and core forward counts must match across paired variants. A mismatch stops the run and preserves the evidence.

Repair during search is inside MOEVA; CAPGD's inner iterations remain unchanged. Extra repair and selection work is recorded separately, so matched core budgets do not imply equal total computation or runtime.

In [ ]:
RUN = SESSION / "experiment"
command = [PYTHON, ROOT / "baseline/run_frozen_followup.py", "--pilot-zip", PILOT_INPUT,
           "--semantics-zip", SEMANTICS_INPUT, "--out", RUN,
           "--repo", ROOT / "vendor/tabularbench", "--assets", ROOT / "assets"]
if RUN_MODE == "smoke":
    command.append("--smoke")
elif RUN_MODE == "preflight":
    command.append("--preflight-only")
run_command(command, "experiment.log")
print("Output:", RUN)

## 4. Recheck the saved evidence and analyze paired results

The existing independent reviewer applies to these same source/final formats. It recomputes native checks, profile checks, distances and model decisions, verifies selection provenance, and checks generation budgets. Intermediate optimizer vectors are not all exported; their logs/digests are cross-checked rather than reconstructed.

Full mode averages the three paired seed differences **within each original record**, then reports a difference in new-evasion rates and an exploratory record-bootstrap interval. Unknown host/window correlations remain a limitation. These are development findings, not an untouched test or real-world IDS accuracy.

In [ ]:
REVIEW = SESSION / "independent_review.json"
ANALYSIS = SESSION / "paired_analysis"
if RUN_MODE != "preflight":
    run_command([PYTHON, ROOT / "scripts/review_notebook08_evidence.py", RUN, REVIEW], "review.log")
    assert json.loads(REVIEW.read_text())["status"] == "passed"
    if RUN_MODE == "full":
        run_command([PYTHON, ROOT / "scripts/summarize_frozen_followup.py", "--run", RUN,
                     "--review", REVIEW, "--out", ANALYSIS], "analysis.log")
else:
    print("Preflight only: no attack results to review or analyze.")

In [ ]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

preflight = json.loads((RUN / "preflight.json").read_text())
display(pd.DataFrame([dict(model=m, selected=128, clean_detected=d["clean_detected"], already_missed=d["already_missed"])
                      for m, d in preflight["models"].items()]))
if RUN_MODE != "preflight":
    comparison = pd.read_csv(RUN / "comparison.csv")
    display(comparison[["model", "seed", "method", "clean_detected", "covered_records", "newly_evaded", "new_evasion_rate"]])
    print("Independent evidence review:", json.loads(REVIEW.read_text())["status"])
if RUN_MODE == "full":
    analysis = json.loads((ANALYSIS / "paired_summary.json").read_text())
    display(pd.DataFrame([dict(model=m, **d["primary_comparison"]) for m, d in analysis["models"].items()]))
    print("Positive differences favor repair during MOEVA; negative differences favor repair afterward.")
    print("Three seeds reuse the same originals. Exploratory intervals do not establish network-level independence.")
else:
    print("IMPLEMENTATION CHECK ONLY. Run full mode in a new session for the fixed research experiment.")

## 5. Download and share the new result ZIP

Share this new ZIP for review. It contains the fixed-input provenance, controls, model/attack evidence, paired analysis (full mode), source snapshots, and logs. The original input ZIPs, datasets, model weights, and environments are excluded.

If a cell fails, run this last cell separately to save partial evidence. Do not filter a problematic original or change a rule just to finish the run. Keep the two earlier ZIPs.

In [ ]:
import zipfile
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
archive = EXPORTS / (unique_name(SESSION.name + "_results") + ".zip")
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(SESSION.rglob("*")):
        if path.is_file() and "upload" not in path.relative_to(SESSION).parts:
            z.write(path, path.relative_to(SESSION.parent))
with zipfile.ZipFile(archive) as z:
    if z.testzip() is not None:
        raise RuntimeError("Export CRC check failed")
print("Result ZIP:", archive)
print("SHA256:", file_hash(archive))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError("Preserving existing archive: " + str(target))
    shutil.copy2(archive, target)
    print("Copied to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))